# Colab Session: img2gps3k-wedetect
Generated from colab-cli history log.

**Session Created**: 2026-08-04 02:30:51
- Endpoint: `gpu-t4-s-kkb-ass1c1-2brww22f18vym`
- Hardware: `T4`

In [ ]:
import torch, sys, os
print(sys.version)
print(torch.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(os.getcwd())


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
2.11.0+cu128
True Tesla T4
/content


### Automation: install (2026-08-04 02:36:37)

In [ ]:

import subprocess, sys
def install():
    packages = ['kagglehub', 'pandas', 'transformers', 'onnxruntime-gpu']
    try:
        subprocess.check_call(['uv', 'pip', 'install', '--system'] + packages)
        print('Installation Complete (via uv)!')
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
        print('Installation Complete (via pip)!')
install()


In [ ]:
# Result of previous automation

Installation Complete (via uv)!


In [ ]:
import os
os.makedirs('/content/wedetect', exist_ok=True)
print('REMOTE_DIRS_READY')


REMOTE_DIRS_READY


*File Operation*: `upload` on `/content/geoclip_source.zip`

*File Operation*: `upload` on `/content/intervention.py`

*File Operation*: `upload` on `/content/wedetect_proposals.py`

*File Operation*: `upload` on `/content/eval_img2gps3k_wedetect_colab.py`

*File Operation*: `upload` on `/content/wedetect/wedetect_anything_base.onnx`

*File Operation*: `upload` on `/content/wedetect/part_000`

*File Operation*: `upload` on `/content/wedetect/part_001`

*File Operation*: `upload` on `/content/wedetect/part_002`

*File Operation*: `upload` on `/content/wedetect/part_003`

*File Operation*: `upload` on `/content/wedetect/part_004`

*File Operation*: `upload` on `/content/wedetect/part_005`

*File Operation*: `upload` on `/content/wedetect/part_006`

In [ ]:
from pathlib import Path
import hashlib
root = Path('/content/wedetect')
target = root / 'wedetect_anything_base.onnx.data'
with target.open('wb') as output:
    for part in sorted(root.glob('part_*')):
        with part.open('rb') as stream:
            while chunk := stream.read(8 * 1024 * 1024):
                output.write(chunk)
h = hashlib.sha256()
with target.open('rb') as stream:
    while chunk := stream.read(8 * 1024 * 1024):
        h.update(chunk)
print('REMOTE_MODEL_DATA', target.stat().st_size, h.hexdigest().upper())


REMOTE_MODEL_DATA 429326336 A6D45CAACD8A5F1CCD142BA11CBC6F70FFD372C18491ACE347C205FC387487AD


In [ ]:
print('PING')


PING


In [ ]:
import os
print('MAX_IMAGES', os.environ.get('MAX_IMAGES'))


MAX_IMAGES None


In [ ]:
import os
os.environ['MAX_IMAGES'] = '5'
os.environ['WEDETECT_SCORE_THRESHOLD'] = '0.4'
print('SMOKE_ENV_READY', os.environ['MAX_IMAGES'])


SMOKE_ENV_READY 5


In [ ]:
"""Evaluate GeoCLIP intervention over every retained WeDetect-Uni proposal.

This script is intended for ``colab exec -f``. Expected remote assets:

* /content/geoclip_source.zip
* /content/intervention.py
* /content/wedetect_proposals.py
* /content/wedetect/wedetect_anything_base.onnx{,.data}

It checkpoints per-image results so a second execution can resume after a
runtime interruption. Set MAX_IMAGES in the remote kernel for a smoke test.
"""

from __future__ import annotations

import json
import os
import sys
import time
import types
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from transformers import AutoProcessor


DATASET_HANDLE = "lbgan2000/imgps3k-yfcc4k-cleaned"
SCORE_THRESHOLD = float(os.getenv("WEDETECT_SCORE_THRESHOLD", "0.4"))
IOU_THRESHOLD = float(os.getenv("WEDETECT_IOU_THRESHOLD", "0.7"))
PROPOSAL_TOP_K = int(os.getenv("WEDETECT_PROPOSAL_TOP_K", "1000"))
LAYER_INDEX = int(os.getenv("INTERVENTION_LAYER", "23"))
ATTENTION_A = float(os.getenv("INTERVENTION_A", "1.0"))
ATTENTION_B = float(os.getenv("INTERVENTION_B", "0.0"))
BASELINE_BATCH_SIZE = int(os.getenv("BASELINE_BATCH_SIZE", "64"))
PROPOSAL_BATCH_SIZE = int(os.getenv("PROPOSAL_BATCH_SIZE", "8"))
MAX_IMAGES = int(os.getenv("MAX_IMAGES", "0"))
CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "25"))

CONTENT = Path("/content")
SOURCE_ZIP = CONTENT / "geoclip_source.zip"
SOURCE_DIR = CONTENT / "geoclip_source"
ONNX_PATH = CONTENT / "wedetect" / "wedetect_anything_base.onnx"
OUTPUT_DIR = CONTENT / "img2gps3k_wedetect_eval"
PER_IMAGE_PATH = OUTPUT_DIR / "per_image.csv"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
THRESHOLDS_KM = (1, 25, 200, 750, 2500)

ImageFile.LOAD_TRUNCATED_IMAGES = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_source() -> None:
    if not SOURCE_DIR.exists():
        if not SOURCE_ZIP.is_file():
            raise FileNotFoundError(f"Missing uploaded source archive: {SOURCE_ZIP}")
        SOURCE_DIR.mkdir(parents=True)
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            archive.extractall(SOURCE_DIR)
    for local_module in ("intervention.py", "wedetect_proposals.py"):
        source = CONTENT / local_module
        target = SOURCE_DIR / local_module
        if source.is_file() and not target.is_file():
            target.write_bytes(source.read_bytes())
    sys.path.insert(0, str(SOURCE_DIR))


def haversine_km(predicted: np.ndarray, target: np.ndarray) -> np.ndarray:
    predicted = np.asarray(predicted, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    lat1, lon1 = np.radians(target[..., 0]), np.radians(target[..., 1])
    lat2, lon2 = np.radians(predicted[..., 0]), np.radians(predicted[..., 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(np.maximum(0, 1 - value)))


def metric_summary(distances: np.ndarray) -> dict:
    distances = np.asarray(distances, dtype=np.float64)
    return {
        "mean_distance_km": float(np.mean(distances)),
        "median_distance_km": float(np.median(distances)),
        "accuracy": {
            f"acc_{threshold}_km": float(np.mean(distances <= threshold))
            for threshold in THRESHOLDS_KM
        },
    }


def enable_batched_masks(state) -> None:
    """Allow one independent proposal mask per image in an inference batch."""

    def key_bias_for_layer(self, layer_idx: int, num_positions: int):
        a, b = self.layer_ab.get(layer_idx, (0.0, 0.0))
        if (a == 0.0 and b == 0.0) or self.in_region_mask is None:
            return None
        mask = self.in_region_mask
        if mask.ndim == 1:
            bias = torch.full((num_positions,), b, dtype=torch.float32, device=mask.device)
            bias[0] = 0.0
            bias[1:].masked_fill_(mask, a)
            return bias
        if mask.ndim != 2 or mask.shape[1] != num_positions - 1:
            raise ValueError(f"Unexpected batched mask shape {tuple(mask.shape)}")
        bias = torch.full(
            (mask.shape[0], num_positions), b, dtype=torch.float32, device=mask.device
        )
        bias[:, 0] = 0.0
        bias[:, 1:].masked_fill_(mask, a)
        return bias[:, None, None, :]

    state.key_bias_for_layer = types.MethodType(key_bias_for_layer, state)


ensure_source()
from geoclip import GeoCLIP  # noqa: E402
import intervention  # noqa: E402
import wedetect_proposals  # noqa: E402


if not torch.cuda.is_available():
    raise RuntimeError("This full evaluation requires a Colab GPU runtime")
device = torch.device("cuda")
print("CONFIG", json.dumps({
    "dataset": DATASET_HANDLE,
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "proposal_top_k": PROPOSAL_TOP_K,
    "layer": LAYER_INDEX,
    "a": ATTENTION_A,
    "b": ATTENTION_B,
    "max_images": MAX_IMAGES,
}, sort_keys=True))
print("GPU", torch.cuda.get_device_name(0))

dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_path / "test_set" / "im2gps3k_places365.csv"
image_dir = dataset_path / "test_set" / "im2gps3ktest" / "im2gps3ktest"
frame = pd.read_csv(csv_path)
frame["image_path"] = frame["name"].map(lambda name: str(image_dir / name))
missing = frame.loc[~frame["image_path"].map(os.path.isfile), "name"].tolist()
if missing:
    raise RuntimeError(f"Missing labeled images: {missing[:10]}")
if MAX_IMAGES > 0:
    frame = frame.iloc[:MAX_IMAGES].copy()
frame = frame.reset_index(drop=True)
targets = frame[["LAT", "LON"]].to_numpy(dtype=np.float64)
print(f"DATASET path={dataset_path} labeled_images={len(frame)}")

setup_start = time.perf_counter()
model = GeoCLIP().to(device).eval()
model.image_encoder.image_processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-large-patch14", use_fast=False
)
state = intervention.patch_vision_tower(model.image_encoder.CLIP)
enable_batched_masks(state)
with torch.inference_mode():
    gallery_features = F.normalize(
        model.location_encoder(model.gps_gallery.to(device)), dim=1
    )
    logit_scale = model.logit_scale.exp()
gallery_cpu = model.gps_gallery.cpu().numpy()
print(f"GEOCLIP_READY seconds={time.perf_counter() - setup_start:.2f}")

proposer = wedetect_proposals.WeDetectUniONNX(model_path=ONNX_PATH)
# Force session creation now so provider failures surface before baseline work.
proposer._get_session()
print("WEDETECT_READY providers=", proposer.providers)


def predict_pixels(pixel_values: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    with torch.inference_mode():
        features = F.normalize(model.image_encoder(pixel_values.to(device)), dim=1)
        logits = logit_scale * (features @ gallery_features.T)
        probabilities = logits.softmax(dim=1)
        confidence, indices = probabilities.max(dim=1)
    return indices.cpu().numpy(), confidence.cpu().numpy()


baseline_start = time.perf_counter()
baseline_indices: list[int] = []
baseline_confidence: list[float] = []
state.layer_ab = {}
state.in_region_mask = None
for start in range(0, len(frame), BASELINE_BATCH_SIZE):
    batch = frame.iloc[start : start + BASELINE_BATCH_SIZE]
    images = []
    for image_path in batch["image_path"]:
        with Image.open(image_path) as source:
            images.append(source.convert("RGB"))
    pixels = model.image_encoder.image_processor(images=images, return_tensors="pt")[
        "pixel_values"
    ]
    indices, confidence = predict_pixels(pixels)
    baseline_indices.extend(indices.tolist())
    baseline_confidence.extend(confidence.tolist())
    done = min(start + BASELINE_BATCH_SIZE, len(frame))
    if done == len(frame) or done % (BASELINE_BATCH_SIZE * 5) == 0:
        print(f"BASELINE {done}/{len(frame)}")

baseline_indices_array = np.asarray(baseline_indices, dtype=np.int64)
baseline_gps = gallery_cpu[baseline_indices_array]
baseline_distances = haversine_km(baseline_gps, targets)
baseline_seconds = time.perf_counter() - baseline_start
print(f"BASELINE_DONE seconds={baseline_seconds:.2f}")

existing_records: dict[str, dict] = {}
if PER_IMAGE_PATH.is_file():
    previous = pd.read_csv(PER_IMAGE_PATH)
    existing_records = {str(row["name"]): row.to_dict() for _, row in previous.iterrows()}
    print(f"RESUME existing={len(existing_records)}")

records: dict[str, dict] = dict(existing_records)
proposal_start = time.perf_counter()
processed_this_run = 0


def save_checkpoint() -> None:
    ordered = [records[name] for name in frame["name"] if name in records]
    pd.DataFrame(ordered).to_csv(PER_IMAGE_PATH, index=False)
    print(f"CHECKPOINT rows={len(ordered)} path={PER_IMAGE_PATH}")


for row_index, row in frame.iterrows():
    name = str(row["name"])
    if name in records:
        continue
    base_gps = baseline_gps[row_index]
    base_distance = float(baseline_distances[row_index])
    base_confidence = float(baseline_confidence[row_index])
    target = targets[row_index]
    record = {
        "name": name,
        "target_lat": float(target[0]),
        "target_lon": float(target[1]),
        "baseline_lat": float(base_gps[0]),
        "baseline_lon": float(base_gps[1]),
        "baseline_confidence": base_confidence,
        "baseline_distance_km": base_distance,
        "proposal_count": 0,
        "top_objectness_lat": float(base_gps[0]),
        "top_objectness_lon": float(base_gps[1]),
        "top_objectness_confidence": base_confidence,
        "top_objectness_distance_km": base_distance,
        "top_objectness_delta_km": 0.0,
        "oracle_proposal_lat": float(base_gps[0]),
        "oracle_proposal_lon": float(base_gps[1]),
        "oracle_proposal_distance_km": base_distance,
        "oracle_proposal_delta_km": 0.0,
        "oracle_with_baseline_distance_km": base_distance,
        "best_proposal_index": -1,
        "best_proposal_objectness": np.nan,
        "error": "",
    }
    try:
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        raw = proposer.generate(
            image,
            score_threshold=SCORE_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            top_k=PROPOSAL_TOP_K,
        )
        width, height = image.size
        proposals = []
        masks = []
        signatures = set()
        for proposal in raw:
            region = proposal.normalized_region(width, height)
            mask = intervention.build_in_region_mask([region], width, height, 16)
            if not mask.any():
                continue
            signature = mask.numpy().tobytes()
            if signature in signatures:
                continue
            signatures.add(signature)
            proposals.append(proposal)
            masks.append(mask)

        record["proposal_count"] = len(masks)
        if masks:
            prediction_indices = []
            prediction_confidence = []
            state.layer_ab = {LAYER_INDEX: (ATTENTION_A, ATTENTION_B)}
            for chunk_start in range(0, len(masks), PROPOSAL_BATCH_SIZE):
                mask_chunk = masks[chunk_start : chunk_start + PROPOSAL_BATCH_SIZE]
                pixels = model.image_encoder.image_processor(
                    images=[image] * len(mask_chunk), return_tensors="pt"
                )["pixel_values"]
                state.in_region_mask = torch.stack(mask_chunk).to(device)
                indices, confidence = predict_pixels(pixels)
                prediction_indices.extend(indices.tolist())
                prediction_confidence.extend(confidence.tolist())
            state.layer_ab = {}
            state.in_region_mask = None

            proposal_gps = gallery_cpu[np.asarray(prediction_indices, dtype=np.int64)]
            repeated_target = np.repeat(target[None, :], len(masks), axis=0)
            proposal_distances = haversine_km(proposal_gps, repeated_target)
            best_index = int(np.argmin(proposal_distances))

            record.update({
                "top_objectness_lat": float(proposal_gps[0, 0]),
                "top_objectness_lon": float(proposal_gps[0, 1]),
                "top_objectness_confidence": float(prediction_confidence[0]),
                "top_objectness_distance_km": float(proposal_distances[0]),
                "top_objectness_delta_km": base_distance - float(proposal_distances[0]),
                "oracle_proposal_lat": float(proposal_gps[best_index, 0]),
                "oracle_proposal_lon": float(proposal_gps[best_index, 1]),
                "oracle_proposal_distance_km": float(proposal_distances[best_index]),
                "oracle_proposal_delta_km": base_distance - float(proposal_distances[best_index]),
                "oracle_with_baseline_distance_km": min(base_distance, float(proposal_distances[best_index])),
                "best_proposal_index": best_index,
                "best_proposal_objectness": float(proposals[best_index].score),
            })
    except Exception as exc:  # Continue the benchmark and report per-image failures.
        state.layer_ab = {}
        state.in_region_mask = None
        record["error"] = repr(exc)
        print(f"IMAGE_ERROR name={name} error={exc!r}")

    records[name] = record
    processed_this_run += 1
    total_done = sum(name in records for name in frame["name"])
    if processed_this_run % CHECKPOINT_EVERY == 0 or total_done == len(frame):
        save_checkpoint()
        elapsed = time.perf_counter() - proposal_start
        print(f"PROPOSALS {total_done}/{len(frame)} elapsed={elapsed:.1f}s")

save_checkpoint()
result_frame = pd.read_csv(PER_IMAGE_PATH)
proposal_seconds = time.perf_counter() - proposal_start
errors = result_frame["error"].fillna("").astype(str)
counts = result_frame["proposal_count"].to_numpy(dtype=np.int64)
summary = {
    "dataset": DATASET_HANDLE,
    "evaluated_images": int(len(result_frame)),
    "failed_images": int(np.sum(errors != "")),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0),
    "wedetect_provider": proposer.providers[0] if proposer.providers else None,
    "config": {
        "score_threshold": SCORE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "proposal_top_k": PROPOSAL_TOP_K,
        "layer": LAYER_INDEX,
        "a": ATTENTION_A,
        "b": ATTENTION_B,
        "baseline_batch_size": BASELINE_BATCH_SIZE,
        "proposal_batch_size": PROPOSAL_BATCH_SIZE,
    },
    "proposal_statistics": {
        "images_with_proposals": int(np.sum(counts > 0)),
        "images_without_proposals": int(np.sum(counts == 0)),
        "total_unique_patch_masks": int(np.sum(counts)),
        "mean_per_image": float(np.mean(counts)),
        "median_per_image": float(np.median(counts)),
        "max_per_image": int(np.max(counts)),
    },
    "metrics": {
        "baseline": metric_summary(result_frame["baseline_distance_km"].to_numpy()),
        "top_objectness": metric_summary(result_frame["top_objectness_distance_km"].to_numpy()),
        "oracle_proposal": metric_summary(result_frame["oracle_proposal_distance_km"].to_numpy()),
        "oracle_with_baseline": metric_summary(result_frame["oracle_with_baseline_distance_km"].to_numpy()),
    },
    "timing_seconds": {
        "baseline": baseline_seconds,
        "proposal_stage_this_run": proposal_seconds,
    },
    "artifacts": {
        "per_image": str(PER_IMAGE_PATH),
        "summary": str(SUMMARY_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("FINAL_SUMMARY")
print(json.dumps(summary, indent=2))


ModuleNotFoundError: No module named 'geoclip'

In [ ]:
from pathlib import Path
for p in list(Path('/content/geoclip_source').rglob('*'))[:30]: print(repr(str(p)))


'/content/geoclip_source/geoclip\\__init__.py'
'/content/geoclip_source/geoclip\\model\\rff\\functional.py'
'/content/geoclip_source/geoclip\\model\\__init__.py'
'/content/geoclip_source/intervention.py'
'/content/geoclip_source/geoclip\\model\\rff\\__pycache__\\__init__.cpython-313.pyc'
'/content/geoclip_source/geoclip\\model\\weights\\image_encoder_mlp_weights.pth'
'/content/geoclip_source/geoclip\\model\\weights\\logit_scale_weights.pth'
'/content/geoclip_source/geoclip\\model\\rff\\__init__.py'
'/content/geoclip_source/geoclip\\model\\__pycache__\\location_encoder.cpython-313.pyc'
'/content/geoclip_source/wedetect_proposals.py'
'/content/geoclip_source/geoclip\\model\\image_encoder.py'
'/content/geoclip_source/geoclip\\train\\dataloader.py'
'/content/geoclip_source/geoclip\\model\\weights\\location_encoder_weights.pth'
'/content/geoclip_source/geoclip\\model\\location_encoder.py'
'/content/geoclip_source/geoclip\\model\\__pycache__\\__init__.cpython-313.pyc'
'/content/geoclip_sourc

In [ ]:
"""Evaluate GeoCLIP intervention over every retained WeDetect-Uni proposal.

This script is intended for ``colab exec -f``. Expected remote assets:

* /content/geoclip_source.zip
* /content/intervention.py
* /content/wedetect_proposals.py
* /content/wedetect/wedetect_anything_base.onnx{,.data}

It checkpoints per-image results so a second execution can resume after a
runtime interruption. Set MAX_IMAGES in the remote kernel for a smoke test.
"""

from __future__ import annotations

import json
import os
import sys
import time
import types
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from transformers import AutoProcessor


DATASET_HANDLE = "lbgan2000/imgps3k-yfcc4k-cleaned"
SCORE_THRESHOLD = float(os.getenv("WEDETECT_SCORE_THRESHOLD", "0.4"))
IOU_THRESHOLD = float(os.getenv("WEDETECT_IOU_THRESHOLD", "0.7"))
PROPOSAL_TOP_K = int(os.getenv("WEDETECT_PROPOSAL_TOP_K", "1000"))
LAYER_INDEX = int(os.getenv("INTERVENTION_LAYER", "23"))
ATTENTION_A = float(os.getenv("INTERVENTION_A", "1.0"))
ATTENTION_B = float(os.getenv("INTERVENTION_B", "0.0"))
BASELINE_BATCH_SIZE = int(os.getenv("BASELINE_BATCH_SIZE", "64"))
PROPOSAL_BATCH_SIZE = int(os.getenv("PROPOSAL_BATCH_SIZE", "8"))
MAX_IMAGES = int(os.getenv("MAX_IMAGES", "0"))
CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "25"))

CONTENT = Path("/content")
SOURCE_ZIP = CONTENT / "geoclip_source.zip"
SOURCE_DIR = CONTENT / "geoclip_source"
ONNX_PATH = CONTENT / "wedetect" / "wedetect_anything_base.onnx"
OUTPUT_DIR = CONTENT / "img2gps3k_wedetect_eval"
PER_IMAGE_PATH = OUTPUT_DIR / "per_image.csv"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
THRESHOLDS_KM = (1, 25, 200, 750, 2500)

ImageFile.LOAD_TRUNCATED_IMAGES = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_source() -> None:
    package_marker = SOURCE_DIR / "geoclip" / "__init__.py"
    if not package_marker.is_file():
        if not SOURCE_ZIP.is_file():
            raise FileNotFoundError(f"Missing uploaded source archive: {SOURCE_ZIP}")
        SOURCE_DIR.mkdir(parents=True)
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            # PowerShell Compress-Archive stores Windows backslashes in member
            # names. Linux's zipfile otherwise extracts those as literal
            # characters instead of directories.
            for member in archive.infolist():
                relative = Path(*member.filename.replace("\\", "/").split("/"))
                target = SOURCE_DIR / relative
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    while chunk := source.read(1024 * 1024):
                        destination.write(chunk)
    for local_module in ("intervention.py", "wedetect_proposals.py"):
        source = CONTENT / local_module
        target = SOURCE_DIR / local_module
        if source.is_file() and not target.is_file():
            target.write_bytes(source.read_bytes())
    sys.path.insert(0, str(SOURCE_DIR))


def haversine_km(predicted: np.ndarray, target: np.ndarray) -> np.ndarray:
    predicted = np.asarray(predicted, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    lat1, lon1 = np.radians(target[..., 0]), np.radians(target[..., 1])
    lat2, lon2 = np.radians(predicted[..., 0]), np.radians(predicted[..., 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(np.maximum(0, 1 - value)))


def metric_summary(distances: np.ndarray) -> dict:
    distances = np.asarray(distances, dtype=np.float64)
    return {
        "mean_distance_km": float(np.mean(distances)),
        "median_distance_km": float(np.median(distances)),
        "accuracy": {
            f"acc_{threshold}_km": float(np.mean(distances <= threshold))
            for threshold in THRESHOLDS_KM
        },
    }


def enable_batched_masks(state) -> None:
    """Allow one independent proposal mask per image in an inference batch."""

    def key_bias_for_layer(self, layer_idx: int, num_positions: int):
        a, b = self.layer_ab.get(layer_idx, (0.0, 0.0))
        if (a == 0.0 and b == 0.0) or self.in_region_mask is None:
            return None
        mask = self.in_region_mask
        if mask.ndim == 1:
            bias = torch.full((num_positions,), b, dtype=torch.float32, device=mask.device)
            bias[0] = 0.0
            bias[1:].masked_fill_(mask, a)
            return bias
        if mask.ndim != 2 or mask.shape[1] != num_positions - 1:
            raise ValueError(f"Unexpected batched mask shape {tuple(mask.shape)}")
        bias = torch.full(
            (mask.shape[0], num_positions), b, dtype=torch.float32, device=mask.device
        )
        bias[:, 0] = 0.0
        bias[:, 1:].masked_fill_(mask, a)
        return bias[:, None, None, :]

    state.key_bias_for_layer = types.MethodType(key_bias_for_layer, state)


ensure_source()
from geoclip import GeoCLIP  # noqa: E402
import intervention  # noqa: E402
import wedetect_proposals  # noqa: E402


if not torch.cuda.is_available():
    raise RuntimeError("This full evaluation requires a Colab GPU runtime")
device = torch.device("cuda")
print("CONFIG", json.dumps({
    "dataset": DATASET_HANDLE,
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "proposal_top_k": PROPOSAL_TOP_K,
    "layer": LAYER_INDEX,
    "a": ATTENTION_A,
    "b": ATTENTION_B,
    "max_images": MAX_IMAGES,
}, sort_keys=True))
print("GPU", torch.cuda.get_device_name(0))

dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_path / "test_set" / "im2gps3k_places365.csv"
image_dir = dataset_path / "test_set" / "im2gps3ktest" / "im2gps3ktest"
frame = pd.read_csv(csv_path)
frame["image_path"] = frame["name"].map(lambda name: str(image_dir / name))
missing = frame.loc[~frame["image_path"].map(os.path.isfile), "name"].tolist()
if missing:
    raise RuntimeError(f"Missing labeled images: {missing[:10]}")
if MAX_IMAGES > 0:
    frame = frame.iloc[:MAX_IMAGES].copy()
frame = frame.reset_index(drop=True)
targets = frame[["LAT", "LON"]].to_numpy(dtype=np.float64)
print(f"DATASET path={dataset_path} labeled_images={len(frame)}")

setup_start = time.perf_counter()
model = GeoCLIP().to(device).eval()
model.image_encoder.image_processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-large-patch14", use_fast=False
)
state = intervention.patch_vision_tower(model.image_encoder.CLIP)
enable_batched_masks(state)
with torch.inference_mode():
    gallery_features = F.normalize(
        model.location_encoder(model.gps_gallery.to(device)), dim=1
    )
    logit_scale = model.logit_scale.exp()
gallery_cpu = model.gps_gallery.cpu().numpy()
print(f"GEOCLIP_READY seconds={time.perf_counter() - setup_start:.2f}")

proposer = wedetect_proposals.WeDetectUniONNX(model_path=ONNX_PATH)
# Force session creation now so provider failures surface before baseline work.
proposer._get_session()
print("WEDETECT_READY providers=", proposer.providers)


def predict_pixels(pixel_values: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    with torch.inference_mode():
        features = F.normalize(model.image_encoder(pixel_values.to(device)), dim=1)
        logits = logit_scale * (features @ gallery_features.T)
        probabilities = logits.softmax(dim=1)
        confidence, indices = probabilities.max(dim=1)
    return indices.cpu().numpy(), confidence.cpu().numpy()


baseline_start = time.perf_counter()
baseline_indices: list[int] = []
baseline_confidence: list[float] = []
state.layer_ab = {}
state.in_region_mask = None
for start in range(0, len(frame), BASELINE_BATCH_SIZE):
    batch = frame.iloc[start : start + BASELINE_BATCH_SIZE]
    images = []
    for image_path in batch["image_path"]:
        with Image.open(image_path) as source:
            images.append(source.convert("RGB"))
    pixels = model.image_encoder.image_processor(images=images, return_tensors="pt")[
        "pixel_values"
    ]
    indices, confidence = predict_pixels(pixels)
    baseline_indices.extend(indices.tolist())
    baseline_confidence.extend(confidence.tolist())
    done = min(start + BASELINE_BATCH_SIZE, len(frame))
    if done == len(frame) or done % (BASELINE_BATCH_SIZE * 5) == 0:
        print(f"BASELINE {done}/{len(frame)}")

baseline_indices_array = np.asarray(baseline_indices, dtype=np.int64)
baseline_gps = gallery_cpu[baseline_indices_array]
baseline_distances = haversine_km(baseline_gps, targets)
baseline_seconds = time.perf_counter() - baseline_start
print(f"BASELINE_DONE seconds={baseline_seconds:.2f}")

existing_records: dict[str, dict] = {}
if PER_IMAGE_PATH.is_file():
    previous = pd.read_csv(PER_IMAGE_PATH)
    existing_records = {str(row["name"]): row.to_dict() for _, row in previous.iterrows()}
    print(f"RESUME existing={len(existing_records)}")

records: dict[str, dict] = dict(existing_records)
proposal_start = time.perf_counter()
processed_this_run = 0


def save_checkpoint() -> None:
    ordered = [records[name] for name in frame["name"] if name in records]
    pd.DataFrame(ordered).to_csv(PER_IMAGE_PATH, index=False)
    print(f"CHECKPOINT rows={len(ordered)} path={PER_IMAGE_PATH}")


for row_index, row in frame.iterrows():
    name = str(row["name"])
    if name in records:
        continue
    base_gps = baseline_gps[row_index]
    base_distance = float(baseline_distances[row_index])
    base_confidence = float(baseline_confidence[row_index])
    target = targets[row_index]
    record = {
        "name": name,
        "target_lat": float(target[0]),
        "target_lon": float(target[1]),
        "baseline_lat": float(base_gps[0]),
        "baseline_lon": float(base_gps[1]),
        "baseline_confidence": base_confidence,
        "baseline_distance_km": base_distance,
        "proposal_count": 0,
        "top_objectness_lat": float(base_gps[0]),
        "top_objectness_lon": float(base_gps[1]),
        "top_objectness_confidence": base_confidence,
        "top_objectness_distance_km": base_distance,
        "top_objectness_delta_km": 0.0,
        "oracle_proposal_lat": float(base_gps[0]),
        "oracle_proposal_lon": float(base_gps[1]),
        "oracle_proposal_distance_km": base_distance,
        "oracle_proposal_delta_km": 0.0,
        "oracle_with_baseline_distance_km": base_distance,
        "best_proposal_index": -1,
        "best_proposal_objectness": np.nan,
        "error": "",
    }
    try:
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        raw = proposer.generate(
            image,
            score_threshold=SCORE_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            top_k=PROPOSAL_TOP_K,
        )
        width, height = image.size
        proposals = []
        masks = []
        signatures = set()
        for proposal in raw:
            region = proposal.normalized_region(width, height)
            mask = intervention.build_in_region_mask([region], width, height, 16)
            if not mask.any():
                continue
            signature = mask.numpy().tobytes()
            if signature in signatures:
                continue
            signatures.add(signature)
            proposals.append(proposal)
            masks.append(mask)

        record["proposal_count"] = len(masks)
        if masks:
            prediction_indices = []
            prediction_confidence = []
            state.layer_ab = {LAYER_INDEX: (ATTENTION_A, ATTENTION_B)}
            for chunk_start in range(0, len(masks), PROPOSAL_BATCH_SIZE):
                mask_chunk = masks[chunk_start : chunk_start + PROPOSAL_BATCH_SIZE]
                pixels = model.image_encoder.image_processor(
                    images=[image] * len(mask_chunk), return_tensors="pt"
                )["pixel_values"]
                state.in_region_mask = torch.stack(mask_chunk).to(device)
                indices, confidence = predict_pixels(pixels)
                prediction_indices.extend(indices.tolist())
                prediction_confidence.extend(confidence.tolist())
            state.layer_ab = {}
            state.in_region_mask = None

            proposal_gps = gallery_cpu[np.asarray(prediction_indices, dtype=np.int64)]
            repeated_target = np.repeat(target[None, :], len(masks), axis=0)
            proposal_distances = haversine_km(proposal_gps, repeated_target)
            best_index = int(np.argmin(proposal_distances))

            record.update({
                "top_objectness_lat": float(proposal_gps[0, 0]),
                "top_objectness_lon": float(proposal_gps[0, 1]),
                "top_objectness_confidence": float(prediction_confidence[0]),
                "top_objectness_distance_km": float(proposal_distances[0]),
                "top_objectness_delta_km": base_distance - float(proposal_distances[0]),
                "oracle_proposal_lat": float(proposal_gps[best_index, 0]),
                "oracle_proposal_lon": float(proposal_gps[best_index, 1]),
                "oracle_proposal_distance_km": float(proposal_distances[best_index]),
                "oracle_proposal_delta_km": base_distance - float(proposal_distances[best_index]),
                "oracle_with_baseline_distance_km": min(base_distance, float(proposal_distances[best_index])),
                "best_proposal_index": best_index,
                "best_proposal_objectness": float(proposals[best_index].score),
            })
    except Exception as exc:  # Continue the benchmark and report per-image failures.
        state.layer_ab = {}
        state.in_region_mask = None
        record["error"] = repr(exc)
        print(f"IMAGE_ERROR name={name} error={exc!r}")

    records[name] = record
    processed_this_run += 1
    total_done = sum(name in records for name in frame["name"])
    if processed_this_run % CHECKPOINT_EVERY == 0 or total_done == len(frame):
        save_checkpoint()
        elapsed = time.perf_counter() - proposal_start
        print(f"PROPOSALS {total_done}/{len(frame)} elapsed={elapsed:.1f}s")

save_checkpoint()
result_frame = pd.read_csv(PER_IMAGE_PATH)
proposal_seconds = time.perf_counter() - proposal_start
errors = result_frame["error"].fillna("").astype(str)
counts = result_frame["proposal_count"].to_numpy(dtype=np.int64)
summary = {
    "dataset": DATASET_HANDLE,
    "evaluated_images": int(len(result_frame)),
    "failed_images": int(np.sum(errors != "")),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0),
    "wedetect_provider": proposer.providers[0] if proposer.providers else None,
    "config": {
        "score_threshold": SCORE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "proposal_top_k": PROPOSAL_TOP_K,
        "layer": LAYER_INDEX,
        "a": ATTENTION_A,
        "b": ATTENTION_B,
        "baseline_batch_size": BASELINE_BATCH_SIZE,
        "proposal_batch_size": PROPOSAL_BATCH_SIZE,
    },
    "proposal_statistics": {
        "images_with_proposals": int(np.sum(counts > 0)),
        "images_without_proposals": int(np.sum(counts == 0)),
        "total_unique_patch_masks": int(np.sum(counts)),
        "mean_per_image": float(np.mean(counts)),
        "median_per_image": float(np.median(counts)),
        "max_per_image": int(np.max(counts)),
    },
    "metrics": {
        "baseline": metric_summary(result_frame["baseline_distance_km"].to_numpy()),
        "top_objectness": metric_summary(result_frame["top_objectness_distance_km"].to_numpy()),
        "oracle_proposal": metric_summary(result_frame["oracle_proposal_distance_km"].to_numpy()),
        "oracle_with_baseline": metric_summary(result_frame["oracle_with_baseline_distance_km"].to_numpy()),
    },
    "timing_seconds": {
        "baseline": baseline_seconds,
        "proposal_stage_this_run": proposal_seconds,
    },
    "artifacts": {
        "per_image": str(PER_IMAGE_PATH),
        "summary": str(SUMMARY_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("FINAL_SUMMARY")
print(json.dumps(summary, indent=2))


FileExistsError: [Errno 17] File exists: '/content/geoclip_source'

In [ ]:
"""Evaluate GeoCLIP intervention over every retained WeDetect-Uni proposal.

This script is intended for ``colab exec -f``. Expected remote assets:

* /content/geoclip_source.zip
* /content/intervention.py
* /content/wedetect_proposals.py
* /content/wedetect/wedetect_anything_base.onnx{,.data}

It checkpoints per-image results so a second execution can resume after a
runtime interruption. Set MAX_IMAGES in the remote kernel for a smoke test.
"""

from __future__ import annotations

import json
import os
import sys
import time
import types
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from transformers import AutoProcessor


DATASET_HANDLE = "lbgan2000/imgps3k-yfcc4k-cleaned"
SCORE_THRESHOLD = float(os.getenv("WEDETECT_SCORE_THRESHOLD", "0.4"))
IOU_THRESHOLD = float(os.getenv("WEDETECT_IOU_THRESHOLD", "0.7"))
PROPOSAL_TOP_K = int(os.getenv("WEDETECT_PROPOSAL_TOP_K", "1000"))
LAYER_INDEX = int(os.getenv("INTERVENTION_LAYER", "23"))
ATTENTION_A = float(os.getenv("INTERVENTION_A", "1.0"))
ATTENTION_B = float(os.getenv("INTERVENTION_B", "0.0"))
BASELINE_BATCH_SIZE = int(os.getenv("BASELINE_BATCH_SIZE", "64"))
PROPOSAL_BATCH_SIZE = int(os.getenv("PROPOSAL_BATCH_SIZE", "8"))
MAX_IMAGES = int(os.getenv("MAX_IMAGES", "0"))
CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "25"))

CONTENT = Path("/content")
SOURCE_ZIP = CONTENT / "geoclip_source.zip"
SOURCE_DIR = CONTENT / "geoclip_source"
ONNX_PATH = CONTENT / "wedetect" / "wedetect_anything_base.onnx"
OUTPUT_DIR = CONTENT / "img2gps3k_wedetect_eval"
PER_IMAGE_PATH = OUTPUT_DIR / "per_image.csv"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
THRESHOLDS_KM = (1, 25, 200, 750, 2500)

ImageFile.LOAD_TRUNCATED_IMAGES = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_source() -> None:
    package_marker = SOURCE_DIR / "geoclip" / "__init__.py"
    if not package_marker.is_file():
        if not SOURCE_ZIP.is_file():
            raise FileNotFoundError(f"Missing uploaded source archive: {SOURCE_ZIP}")
        SOURCE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            # PowerShell Compress-Archive stores Windows backslashes in member
            # names. Linux's zipfile otherwise extracts those as literal
            # characters instead of directories.
            for member in archive.infolist():
                relative = Path(*member.filename.replace("\\", "/").split("/"))
                target = SOURCE_DIR / relative
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    while chunk := source.read(1024 * 1024):
                        destination.write(chunk)
    for local_module in ("intervention.py", "wedetect_proposals.py"):
        source = CONTENT / local_module
        target = SOURCE_DIR / local_module
        if source.is_file() and not target.is_file():
            target.write_bytes(source.read_bytes())
    sys.path.insert(0, str(SOURCE_DIR))


def haversine_km(predicted: np.ndarray, target: np.ndarray) -> np.ndarray:
    predicted = np.asarray(predicted, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    lat1, lon1 = np.radians(target[..., 0]), np.radians(target[..., 1])
    lat2, lon2 = np.radians(predicted[..., 0]), np.radians(predicted[..., 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(np.maximum(0, 1 - value)))


def metric_summary(distances: np.ndarray) -> dict:
    distances = np.asarray(distances, dtype=np.float64)
    return {
        "mean_distance_km": float(np.mean(distances)),
        "median_distance_km": float(np.median(distances)),
        "accuracy": {
            f"acc_{threshold}_km": float(np.mean(distances <= threshold))
            for threshold in THRESHOLDS_KM
        },
    }


def enable_batched_masks(state) -> None:
    """Allow one independent proposal mask per image in an inference batch."""

    def key_bias_for_layer(self, layer_idx: int, num_positions: int):
        a, b = self.layer_ab.get(layer_idx, (0.0, 0.0))
        if (a == 0.0 and b == 0.0) or self.in_region_mask is None:
            return None
        mask = self.in_region_mask
        if mask.ndim == 1:
            bias = torch.full((num_positions,), b, dtype=torch.float32, device=mask.device)
            bias[0] = 0.0
            bias[1:].masked_fill_(mask, a)
            return bias
        if mask.ndim != 2 or mask.shape[1] != num_positions - 1:
            raise ValueError(f"Unexpected batched mask shape {tuple(mask.shape)}")
        bias = torch.full(
            (mask.shape[0], num_positions), b, dtype=torch.float32, device=mask.device
        )
        bias[:, 0] = 0.0
        bias[:, 1:].masked_fill_(mask, a)
        return bias[:, None, None, :]

    state.key_bias_for_layer = types.MethodType(key_bias_for_layer, state)


ensure_source()
from geoclip import GeoCLIP  # noqa: E402
import intervention  # noqa: E402
import wedetect_proposals  # noqa: E402


if not torch.cuda.is_available():
    raise RuntimeError("This full evaluation requires a Colab GPU runtime")
device = torch.device("cuda")
print("CONFIG", json.dumps({
    "dataset": DATASET_HANDLE,
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "proposal_top_k": PROPOSAL_TOP_K,
    "layer": LAYER_INDEX,
    "a": ATTENTION_A,
    "b": ATTENTION_B,
    "max_images": MAX_IMAGES,
}, sort_keys=True))
print("GPU", torch.cuda.get_device_name(0))

dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_path / "test_set" / "im2gps3k_places365.csv"
image_dir = dataset_path / "test_set" / "im2gps3ktest" / "im2gps3ktest"
frame = pd.read_csv(csv_path)
frame["image_path"] = frame["name"].map(lambda name: str(image_dir / name))
missing = frame.loc[~frame["image_path"].map(os.path.isfile), "name"].tolist()
if missing:
    raise RuntimeError(f"Missing labeled images: {missing[:10]}")
if MAX_IMAGES > 0:
    frame = frame.iloc[:MAX_IMAGES].copy()
frame = frame.reset_index(drop=True)
targets = frame[["LAT", "LON"]].to_numpy(dtype=np.float64)
print(f"DATASET path={dataset_path} labeled_images={len(frame)}")

setup_start = time.perf_counter()
model = GeoCLIP().to(device).eval()
model.image_encoder.image_processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-large-patch14", use_fast=False
)
state = intervention.patch_vision_tower(model.image_encoder.CLIP)
enable_batched_masks(state)
with torch.inference_mode():
    gallery_features = F.normalize(
        model.location_encoder(model.gps_gallery.to(device)), dim=1
    )
    logit_scale = model.logit_scale.exp()
gallery_cpu = model.gps_gallery.cpu().numpy()
print(f"GEOCLIP_READY seconds={time.perf_counter() - setup_start:.2f}")

proposer = wedetect_proposals.WeDetectUniONNX(model_path=ONNX_PATH)
# Force session creation now so provider failures surface before baseline work.
proposer._get_session()
print("WEDETECT_READY providers=", proposer.providers)


def predict_pixels(pixel_values: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    with torch.inference_mode():
        features = F.normalize(model.image_encoder(pixel_values.to(device)), dim=1)
        logits = logit_scale * (features @ gallery_features.T)
        probabilities = logits.softmax(dim=1)
        confidence, indices = probabilities.max(dim=1)
    return indices.cpu().numpy(), confidence.cpu().numpy()


baseline_start = time.perf_counter()
baseline_indices: list[int] = []
baseline_confidence: list[float] = []
state.layer_ab = {}
state.in_region_mask = None
for start in range(0, len(frame), BASELINE_BATCH_SIZE):
    batch = frame.iloc[start : start + BASELINE_BATCH_SIZE]
    images = []
    for image_path in batch["image_path"]:
        with Image.open(image_path) as source:
            images.append(source.convert("RGB"))
    pixels = model.image_encoder.image_processor(images=images, return_tensors="pt")[
        "pixel_values"
    ]
    indices, confidence = predict_pixels(pixels)
    baseline_indices.extend(indices.tolist())
    baseline_confidence.extend(confidence.tolist())
    done = min(start + BASELINE_BATCH_SIZE, len(frame))
    if done == len(frame) or done % (BASELINE_BATCH_SIZE * 5) == 0:
        print(f"BASELINE {done}/{len(frame)}")

baseline_indices_array = np.asarray(baseline_indices, dtype=np.int64)
baseline_gps = gallery_cpu[baseline_indices_array]
baseline_distances = haversine_km(baseline_gps, targets)
baseline_seconds = time.perf_counter() - baseline_start
print(f"BASELINE_DONE seconds={baseline_seconds:.2f}")

existing_records: dict[str, dict] = {}
if PER_IMAGE_PATH.is_file():
    previous = pd.read_csv(PER_IMAGE_PATH)
    existing_records = {str(row["name"]): row.to_dict() for _, row in previous.iterrows()}
    print(f"RESUME existing={len(existing_records)}")

records: dict[str, dict] = dict(existing_records)
proposal_start = time.perf_counter()
processed_this_run = 0


def save_checkpoint() -> None:
    ordered = [records[name] for name in frame["name"] if name in records]
    pd.DataFrame(ordered).to_csv(PER_IMAGE_PATH, index=False)
    print(f"CHECKPOINT rows={len(ordered)} path={PER_IMAGE_PATH}")


for row_index, row in frame.iterrows():
    name = str(row["name"])
    if name in records:
        continue
    base_gps = baseline_gps[row_index]
    base_distance = float(baseline_distances[row_index])
    base_confidence = float(baseline_confidence[row_index])
    target = targets[row_index]
    record = {
        "name": name,
        "target_lat": float(target[0]),
        "target_lon": float(target[1]),
        "baseline_lat": float(base_gps[0]),
        "baseline_lon": float(base_gps[1]),
        "baseline_confidence": base_confidence,
        "baseline_distance_km": base_distance,
        "proposal_count": 0,
        "top_objectness_lat": float(base_gps[0]),
        "top_objectness_lon": float(base_gps[1]),
        "top_objectness_confidence": base_confidence,
        "top_objectness_distance_km": base_distance,
        "top_objectness_delta_km": 0.0,
        "oracle_proposal_lat": float(base_gps[0]),
        "oracle_proposal_lon": float(base_gps[1]),
        "oracle_proposal_distance_km": base_distance,
        "oracle_proposal_delta_km": 0.0,
        "oracle_with_baseline_distance_km": base_distance,
        "best_proposal_index": -1,
        "best_proposal_objectness": np.nan,
        "error": "",
    }
    try:
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        raw = proposer.generate(
            image,
            score_threshold=SCORE_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            top_k=PROPOSAL_TOP_K,
        )
        width, height = image.size
        proposals = []
        masks = []
        signatures = set()
        for proposal in raw:
            region = proposal.normalized_region(width, height)
            mask = intervention.build_in_region_mask([region], width, height, 16)
            if not mask.any():
                continue
            signature = mask.numpy().tobytes()
            if signature in signatures:
                continue
            signatures.add(signature)
            proposals.append(proposal)
            masks.append(mask)

        record["proposal_count"] = len(masks)
        if masks:
            prediction_indices = []
            prediction_confidence = []
            state.layer_ab = {LAYER_INDEX: (ATTENTION_A, ATTENTION_B)}
            for chunk_start in range(0, len(masks), PROPOSAL_BATCH_SIZE):
                mask_chunk = masks[chunk_start : chunk_start + PROPOSAL_BATCH_SIZE]
                pixels = model.image_encoder.image_processor(
                    images=[image] * len(mask_chunk), return_tensors="pt"
                )["pixel_values"]
                state.in_region_mask = torch.stack(mask_chunk).to(device)
                indices, confidence = predict_pixels(pixels)
                prediction_indices.extend(indices.tolist())
                prediction_confidence.extend(confidence.tolist())
            state.layer_ab = {}
            state.in_region_mask = None

            proposal_gps = gallery_cpu[np.asarray(prediction_indices, dtype=np.int64)]
            repeated_target = np.repeat(target[None, :], len(masks), axis=0)
            proposal_distances = haversine_km(proposal_gps, repeated_target)
            best_index = int(np.argmin(proposal_distances))

            record.update({
                "top_objectness_lat": float(proposal_gps[0, 0]),
                "top_objectness_lon": float(proposal_gps[0, 1]),
                "top_objectness_confidence": float(prediction_confidence[0]),
                "top_objectness_distance_km": float(proposal_distances[0]),
                "top_objectness_delta_km": base_distance - float(proposal_distances[0]),
                "oracle_proposal_lat": float(proposal_gps[best_index, 0]),
                "oracle_proposal_lon": float(proposal_gps[best_index, 1]),
                "oracle_proposal_distance_km": float(proposal_distances[best_index]),
                "oracle_proposal_delta_km": base_distance - float(proposal_distances[best_index]),
                "oracle_with_baseline_distance_km": min(base_distance, float(proposal_distances[best_index])),
                "best_proposal_index": best_index,
                "best_proposal_objectness": float(proposals[best_index].score),
            })
    except Exception as exc:  # Continue the benchmark and report per-image failures.
        state.layer_ab = {}
        state.in_region_mask = None
        record["error"] = repr(exc)
        print(f"IMAGE_ERROR name={name} error={exc!r}")

    records[name] = record
    processed_this_run += 1
    total_done = sum(name in records for name in frame["name"])
    if processed_this_run % CHECKPOINT_EVERY == 0 or total_done == len(frame):
        save_checkpoint()
        elapsed = time.perf_counter() - proposal_start
        print(f"PROPOSALS {total_done}/{len(frame)} elapsed={elapsed:.1f}s")

save_checkpoint()
result_frame = pd.read_csv(PER_IMAGE_PATH)
proposal_seconds = time.perf_counter() - proposal_start
errors = result_frame["error"].fillna("").astype(str)
counts = result_frame["proposal_count"].to_numpy(dtype=np.int64)
summary = {
    "dataset": DATASET_HANDLE,
    "evaluated_images": int(len(result_frame)),
    "failed_images": int(np.sum(errors != "")),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0),
    "wedetect_provider": proposer.providers[0] if proposer.providers else None,
    "config": {
        "score_threshold": SCORE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "proposal_top_k": PROPOSAL_TOP_K,
        "layer": LAYER_INDEX,
        "a": ATTENTION_A,
        "b": ATTENTION_B,
        "baseline_batch_size": BASELINE_BATCH_SIZE,
        "proposal_batch_size": PROPOSAL_BATCH_SIZE,
    },
    "proposal_statistics": {
        "images_with_proposals": int(np.sum(counts > 0)),
        "images_without_proposals": int(np.sum(counts == 0)),
        "total_unique_patch_masks": int(np.sum(counts)),
        "mean_per_image": float(np.mean(counts)),
        "median_per_image": float(np.median(counts)),
        "max_per_image": int(np.max(counts)),
    },
    "metrics": {
        "baseline": metric_summary(result_frame["baseline_distance_km"].to_numpy()),
        "top_objectness": metric_summary(result_frame["top_objectness_distance_km"].to_numpy()),
        "oracle_proposal": metric_summary(result_frame["oracle_proposal_distance_km"].to_numpy()),
        "oracle_with_baseline": metric_summary(result_frame["oracle_with_baseline_distance_km"].to_numpy()),
    },
    "timing_seconds": {
        "baseline": baseline_seconds,
        "proposal_stage_this_run": proposal_seconds,
    },
    "artifacts": {
        "per_image": str(PER_IMAGE_PATH),
        "summary": str(SUMMARY_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("FINAL_SUMMARY")
print(json.dumps(summary, indent=2))


CONFIG {"a": 1.0, "b": 0.0, "dataset": "lbgan2000/imgps3k-yfcc4k-cleaned", "iou_threshold": 0.7, "layer": 23, "max_images": 5, "proposal_top_k": 1000, "score_threshold": 0.4}
GPU Tesla T4


  0%|          | 0.00/1.50G [00:00<?, ?B/s]

  0%|          | 1.00M/1.50G [00:01<26:04, 1.03MB/s]

  0%|          | 2.00M/1.50G [00:01<13:21, 2.01MB/s]

  0%|          | 4.00M/1.50G [00:01<06:20, 4.23MB/s]

...

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


GEOCLIP_READY seconds=47.11


WEDETECT_READY providers= ['CPUExecutionProvider']


BASELINE 5/5
BASELINE_DONE seconds=1.19


CHECKPOINT rows=5 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 5/5 elapsed=26.2s
CHECKPOINT rows=5 path=/content/img2gps3k_wedetect_eval/per_image.csv
FINAL_SUMMARY
{
  "dataset": "lbgan2000/imgps3k-yfcc4k-cleaned",
  "evaluated_images": 5,
  "failed_images": 0,
  "device": "cuda",
  "gpu": "Tesla T4",
  "wedetect_provider": "CPUExecutionProvider",
  "config": {
    "score_threshold": 0.4,
    "iou_threshold": 0.7,
    "proposal_top_k": 1000,
    "layer": 23,
    "a": 1.0,
    "b": 0.0,
    "baseline_batch_size": 64,
    "proposal_batch_size": 8
  },
  "proposal_statistics": {
    "images_with_proposals": 5,
    "images_without_proposals": 0,
    "total_unique_patch_masks": 15,
    "mean_per_image": 3.0,
    "median_per_image": 2.0,
    "max_per_image": 6
  },
  "metrics": {
    "baseline": {
      "mean_distance_km": 2722.053949596989,
      "median_distance_km": 2569.749358537205,
      "accuracy": {
        "acc_1_km": 0.2,
        "acc_25_km": 0.2,
        "acc_200

In [ ]:
import onnxruntime as ort, importlib.metadata as md
print('ort', ort.__version__, ort.__file__)
print('providers', ort.get_available_providers())
for name in ('onnxruntime', 'onnxruntime-gpu'):
    try: print(name, md.version(name))
    except Exception as e: print(name, None)


ort 1.28.0 /usr/local/lib/python3.12/dist-packages/onnxruntime/__init__.py
providers ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
onnxruntime None
onnxruntime-gpu 1.28.0


### Automation: install (2026-08-04 02:57:26)

In [ ]:

import subprocess, sys
def install():
    packages = ['onnxruntime-gpu==1.26.0']
    try:
        subprocess.check_call(['uv', 'pip', 'install', '--system'] + packages)
        print('Installation Complete (via uv)!')
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
        print('Installation Complete (via pip)!')
install()


In [ ]:
# Result of previous automation

Installation Complete (via uv)!


In [ ]:
import time, torch, onnxruntime as ort
print('ort', ort.__version__, ort.get_available_providers())
t0=time.time()
s=ort.InferenceSession('/content/wedetect/wedetect_anything_base.onnx', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
print('session', s.get_providers(), 'seconds', time.time()-t0)


ort 1.26.0 ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


session ['CUDAExecutionProvider', 'CPUExecutionProvider'] seconds 0.689892053604126


*File Operation*: `download` on `/content/img2gps3k_wedetect_eval/per_image.csv`

*File Operation*: `download` on `/content/img2gps3k_wedetect_eval/per_image.csv`

*File Operation*: `download` on `/content/img2gps3k_wedetect_eval/per_image.csv`

In [ ]:
"""Evaluate GeoCLIP intervention over every retained WeDetect-Uni proposal.

This script is intended for ``colab exec -f``. Expected remote assets:

* /content/geoclip_source.zip
* /content/intervention.py
* /content/wedetect_proposals.py
* /content/wedetect/wedetect_anything_base.onnx{,.data}

It checkpoints per-image results so a second execution can resume after a
runtime interruption. Set MAX_IMAGES in the remote kernel for a smoke test.
"""

from __future__ import annotations

import json
import os
import sys
import time
import types
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from transformers import AutoProcessor


DATASET_HANDLE = "lbgan2000/imgps3k-yfcc4k-cleaned"
SCORE_THRESHOLD = float(os.getenv("WEDETECT_SCORE_THRESHOLD", "0.4"))
IOU_THRESHOLD = float(os.getenv("WEDETECT_IOU_THRESHOLD", "0.7"))
PROPOSAL_TOP_K = int(os.getenv("WEDETECT_PROPOSAL_TOP_K", "1000"))
LAYER_INDEX = int(os.getenv("INTERVENTION_LAYER", "23"))
ATTENTION_A = float(os.getenv("INTERVENTION_A", "1.0"))
ATTENTION_B = float(os.getenv("INTERVENTION_B", "0.0"))
BASELINE_BATCH_SIZE = int(os.getenv("BASELINE_BATCH_SIZE", "64"))
PROPOSAL_BATCH_SIZE = int(os.getenv("PROPOSAL_BATCH_SIZE", "8"))
MAX_IMAGES = int(os.getenv("MAX_IMAGES", "0"))
CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "25"))

CONTENT = Path("/content")
SOURCE_ZIP = CONTENT / "geoclip_source.zip"
SOURCE_DIR = CONTENT / "geoclip_source"
ONNX_PATH = CONTENT / "wedetect" / "wedetect_anything_base.onnx"
OUTPUT_DIR = CONTENT / "img2gps3k_wedetect_eval"
PER_IMAGE_PATH = OUTPUT_DIR / "per_image.csv"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
THRESHOLDS_KM = (1, 25, 200, 750, 2500)

ImageFile.LOAD_TRUNCATED_IMAGES = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_source() -> None:
    package_marker = SOURCE_DIR / "geoclip" / "__init__.py"
    if not package_marker.is_file():
        if not SOURCE_ZIP.is_file():
            raise FileNotFoundError(f"Missing uploaded source archive: {SOURCE_ZIP}")
        SOURCE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            # PowerShell Compress-Archive stores Windows backslashes in member
            # names. Linux's zipfile otherwise extracts those as literal
            # characters instead of directories.
            for member in archive.infolist():
                relative = Path(*member.filename.replace("\\", "/").split("/"))
                target = SOURCE_DIR / relative
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    while chunk := source.read(1024 * 1024):
                        destination.write(chunk)
    for local_module in ("intervention.py", "wedetect_proposals.py"):
        source = CONTENT / local_module
        target = SOURCE_DIR / local_module
        if source.is_file() and not target.is_file():
            target.write_bytes(source.read_bytes())
    sys.path.insert(0, str(SOURCE_DIR))


def haversine_km(predicted: np.ndarray, target: np.ndarray) -> np.ndarray:
    predicted = np.asarray(predicted, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    lat1, lon1 = np.radians(target[..., 0]), np.radians(target[..., 1])
    lat2, lon2 = np.radians(predicted[..., 0]), np.radians(predicted[..., 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(np.maximum(0, 1 - value)))


def metric_summary(distances: np.ndarray) -> dict:
    distances = np.asarray(distances, dtype=np.float64)
    return {
        "mean_distance_km": float(np.mean(distances)),
        "median_distance_km": float(np.median(distances)),
        "accuracy": {
            f"acc_{threshold}_km": float(np.mean(distances <= threshold))
            for threshold in THRESHOLDS_KM
        },
    }


def enable_batched_masks(state) -> None:
    """Allow one independent proposal mask per image in an inference batch."""

    def key_bias_for_layer(self, layer_idx: int, num_positions: int):
        a, b = self.layer_ab.get(layer_idx, (0.0, 0.0))
        if (a == 0.0 and b == 0.0) or self.in_region_mask is None:
            return None
        mask = self.in_region_mask
        if mask.ndim == 1:
            bias = torch.full((num_positions,), b, dtype=torch.float32, device=mask.device)
            bias[0] = 0.0
            bias[1:].masked_fill_(mask, a)
            return bias
        if mask.ndim != 2 or mask.shape[1] != num_positions - 1:
            raise ValueError(f"Unexpected batched mask shape {tuple(mask.shape)}")
        bias = torch.full(
            (mask.shape[0], num_positions), b, dtype=torch.float32, device=mask.device
        )
        bias[:, 0] = 0.0
        bias[:, 1:].masked_fill_(mask, a)
        return bias[:, None, None, :]

    state.key_bias_for_layer = types.MethodType(key_bias_for_layer, state)


ensure_source()
from geoclip import GeoCLIP  # noqa: E402
import intervention  # noqa: E402
import wedetect_proposals  # noqa: E402


if not torch.cuda.is_available():
    raise RuntimeError("This full evaluation requires a Colab GPU runtime")
device = torch.device("cuda")
print("CONFIG", json.dumps({
    "dataset": DATASET_HANDLE,
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "proposal_top_k": PROPOSAL_TOP_K,
    "layer": LAYER_INDEX,
    "a": ATTENTION_A,
    "b": ATTENTION_B,
    "max_images": MAX_IMAGES,
}, sort_keys=True))
print("GPU", torch.cuda.get_device_name(0))

dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_path / "test_set" / "im2gps3k_places365.csv"
image_dir = dataset_path / "test_set" / "im2gps3ktest" / "im2gps3ktest"
frame = pd.read_csv(csv_path)
frame["image_path"] = frame["name"].map(lambda name: str(image_dir / name))
missing = frame.loc[~frame["image_path"].map(os.path.isfile), "name"].tolist()
if missing:
    raise RuntimeError(f"Missing labeled images: {missing[:10]}")
if MAX_IMAGES > 0:
    frame = frame.iloc[:MAX_IMAGES].copy()
frame = frame.reset_index(drop=True)
targets = frame[["LAT", "LON"]].to_numpy(dtype=np.float64)
print(f"DATASET path={dataset_path} labeled_images={len(frame)}")

setup_start = time.perf_counter()
model = GeoCLIP().to(device).eval()
model.image_encoder.image_processor = AutoProcessor.from_pretrained(
    "openai/clip-vit-large-patch14", use_fast=False
)
state = intervention.patch_vision_tower(model.image_encoder.CLIP)
enable_batched_masks(state)
with torch.inference_mode():
    gallery_features = F.normalize(
        model.location_encoder(model.gps_gallery.to(device)), dim=1
    )
    logit_scale = model.logit_scale.exp()
gallery_cpu = model.gps_gallery.cpu().numpy()
print(f"GEOCLIP_READY seconds={time.perf_counter() - setup_start:.2f}")

proposer = wedetect_proposals.WeDetectUniONNX(model_path=ONNX_PATH)
# Force session creation now so provider failures surface before baseline work.
proposer._get_session()
print("WEDETECT_READY providers=", proposer.providers)


def predict_pixels(pixel_values: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    with torch.inference_mode():
        features = F.normalize(model.image_encoder(pixel_values.to(device)), dim=1)
        logits = logit_scale * (features @ gallery_features.T)
        probabilities = logits.softmax(dim=1)
        confidence, indices = probabilities.max(dim=1)
    return indices.cpu().numpy(), confidence.cpu().numpy()


baseline_start = time.perf_counter()
baseline_indices: list[int] = []
baseline_confidence: list[float] = []
state.layer_ab = {}
state.in_region_mask = None
for start in range(0, len(frame), BASELINE_BATCH_SIZE):
    batch = frame.iloc[start : start + BASELINE_BATCH_SIZE]
    images = []
    for image_path in batch["image_path"]:
        with Image.open(image_path) as source:
            images.append(source.convert("RGB"))
    pixels = model.image_encoder.image_processor(images=images, return_tensors="pt")[
        "pixel_values"
    ]
    indices, confidence = predict_pixels(pixels)
    baseline_indices.extend(indices.tolist())
    baseline_confidence.extend(confidence.tolist())
    done = min(start + BASELINE_BATCH_SIZE, len(frame))
    if done == len(frame) or done % (BASELINE_BATCH_SIZE * 5) == 0:
        print(f"BASELINE {done}/{len(frame)}")

baseline_indices_array = np.asarray(baseline_indices, dtype=np.int64)
baseline_gps = gallery_cpu[baseline_indices_array]
baseline_distances = haversine_km(baseline_gps, targets)
baseline_seconds = time.perf_counter() - baseline_start
print(f"BASELINE_DONE seconds={baseline_seconds:.2f}")

existing_records: dict[str, dict] = {}
if PER_IMAGE_PATH.is_file():
    previous = pd.read_csv(PER_IMAGE_PATH)
    existing_records = {str(row["name"]): row.to_dict() for _, row in previous.iterrows()}
    print(f"RESUME existing={len(existing_records)}")

records: dict[str, dict] = dict(existing_records)
proposal_start = time.perf_counter()
processed_this_run = 0


def save_checkpoint() -> None:
    ordered = [records[name] for name in frame["name"] if name in records]
    pd.DataFrame(ordered).to_csv(PER_IMAGE_PATH, index=False)
    print(f"CHECKPOINT rows={len(ordered)} path={PER_IMAGE_PATH}")


for row_index, row in frame.iterrows():
    name = str(row["name"])
    if name in records:
        continue
    base_gps = baseline_gps[row_index]
    base_distance = float(baseline_distances[row_index])
    base_confidence = float(baseline_confidence[row_index])
    target = targets[row_index]
    record = {
        "name": name,
        "target_lat": float(target[0]),
        "target_lon": float(target[1]),
        "baseline_lat": float(base_gps[0]),
        "baseline_lon": float(base_gps[1]),
        "baseline_confidence": base_confidence,
        "baseline_distance_km": base_distance,
        "proposal_count": 0,
        "top_objectness_lat": float(base_gps[0]),
        "top_objectness_lon": float(base_gps[1]),
        "top_objectness_confidence": base_confidence,
        "top_objectness_distance_km": base_distance,
        "top_objectness_delta_km": 0.0,
        "oracle_proposal_lat": float(base_gps[0]),
        "oracle_proposal_lon": float(base_gps[1]),
        "oracle_proposal_distance_km": base_distance,
        "oracle_proposal_delta_km": 0.0,
        "oracle_with_baseline_distance_km": base_distance,
        "best_proposal_index": -1,
        "best_proposal_objectness": np.nan,
        "error": "",
    }
    try:
        with Image.open(row["image_path"]) as source:
            image = source.convert("RGB")
        raw = proposer.generate(
            image,
            score_threshold=SCORE_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            top_k=PROPOSAL_TOP_K,
        )
        width, height = image.size
        proposals = []
        masks = []
        signatures = set()
        for proposal in raw:
            region = proposal.normalized_region(width, height)
            mask = intervention.build_in_region_mask([region], width, height, 16)
            if not mask.any():
                continue
            signature = mask.numpy().tobytes()
            if signature in signatures:
                continue
            signatures.add(signature)
            proposals.append(proposal)
            masks.append(mask)

        record["proposal_count"] = len(masks)
        if masks:
            prediction_indices = []
            prediction_confidence = []
            state.layer_ab = {LAYER_INDEX: (ATTENTION_A, ATTENTION_B)}
            for chunk_start in range(0, len(masks), PROPOSAL_BATCH_SIZE):
                mask_chunk = masks[chunk_start : chunk_start + PROPOSAL_BATCH_SIZE]
                pixels = model.image_encoder.image_processor(
                    images=[image] * len(mask_chunk), return_tensors="pt"
                )["pixel_values"]
                state.in_region_mask = torch.stack(mask_chunk).to(device)
                indices, confidence = predict_pixels(pixels)
                prediction_indices.extend(indices.tolist())
                prediction_confidence.extend(confidence.tolist())
            state.layer_ab = {}
            state.in_region_mask = None

            proposal_gps = gallery_cpu[np.asarray(prediction_indices, dtype=np.int64)]
            repeated_target = np.repeat(target[None, :], len(masks), axis=0)
            proposal_distances = haversine_km(proposal_gps, repeated_target)
            best_index = int(np.argmin(proposal_distances))

            record.update({
                "top_objectness_lat": float(proposal_gps[0, 0]),
                "top_objectness_lon": float(proposal_gps[0, 1]),
                "top_objectness_confidence": float(prediction_confidence[0]),
                "top_objectness_distance_km": float(proposal_distances[0]),
                "top_objectness_delta_km": base_distance - float(proposal_distances[0]),
                "oracle_proposal_lat": float(proposal_gps[best_index, 0]),
                "oracle_proposal_lon": float(proposal_gps[best_index, 1]),
                "oracle_proposal_distance_km": float(proposal_distances[best_index]),
                "oracle_proposal_delta_km": base_distance - float(proposal_distances[best_index]),
                "oracle_with_baseline_distance_km": min(base_distance, float(proposal_distances[best_index])),
                "best_proposal_index": best_index,
                "best_proposal_objectness": float(proposals[best_index].score),
            })
    except Exception as exc:  # Continue the benchmark and report per-image failures.
        state.layer_ab = {}
        state.in_region_mask = None
        record["error"] = repr(exc)
        print(f"IMAGE_ERROR name={name} error={exc!r}")

    records[name] = record
    processed_this_run += 1
    total_done = sum(name in records for name in frame["name"])
    if processed_this_run % CHECKPOINT_EVERY == 0 or total_done == len(frame):
        save_checkpoint()
        elapsed = time.perf_counter() - proposal_start
        print(f"PROPOSALS {total_done}/{len(frame)} elapsed={elapsed:.1f}s")

save_checkpoint()
result_frame = pd.read_csv(PER_IMAGE_PATH)
proposal_seconds = time.perf_counter() - proposal_start
errors = result_frame["error"].fillna("").astype(str)
counts = result_frame["proposal_count"].to_numpy(dtype=np.int64)
summary = {
    "dataset": DATASET_HANDLE,
    "evaluated_images": int(len(result_frame)),
    "failed_images": int(np.sum(errors != "")),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0),
    "wedetect_provider": proposer.providers[0] if proposer.providers else None,
    "config": {
        "score_threshold": SCORE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "proposal_top_k": PROPOSAL_TOP_K,
        "layer": LAYER_INDEX,
        "a": ATTENTION_A,
        "b": ATTENTION_B,
        "baseline_batch_size": BASELINE_BATCH_SIZE,
        "proposal_batch_size": PROPOSAL_BATCH_SIZE,
    },
    "proposal_statistics": {
        "images_with_proposals": int(np.sum(counts > 0)),
        "images_without_proposals": int(np.sum(counts == 0)),
        "total_unique_patch_masks": int(np.sum(counts)),
        "mean_per_image": float(np.mean(counts)),
        "median_per_image": float(np.median(counts)),
        "max_per_image": int(np.max(counts)),
    },
    "metrics": {
        "baseline": metric_summary(result_frame["baseline_distance_km"].to_numpy()),
        "top_objectness": metric_summary(result_frame["top_objectness_distance_km"].to_numpy()),
        "oracle_proposal": metric_summary(result_frame["oracle_proposal_distance_km"].to_numpy()),
        "oracle_with_baseline": metric_summary(result_frame["oracle_with_baseline_distance_km"].to_numpy()),
    },
    "timing_seconds": {
        "baseline": baseline_seconds,
        "proposal_stage_this_run": proposal_seconds,
    },
    "artifacts": {
        "per_image": str(PER_IMAGE_PATH),
        "summary": str(SUMMARY_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("FINAL_SUMMARY")
print(json.dumps(summary, indent=2))


CONFIG {"a": 1.0, "b": 0.0, "dataset": "lbgan2000/imgps3k-yfcc4k-cleaned", "iou_threshold": 0.7, "layer": 23, "max_images": 0, "proposal_top_k": 1000, "score_threshold": 0.4}
GPU Tesla T4


Using Colab cache for faster access to the 'imgps3k-yfcc4k-cleaned' dataset.


DATASET path=/kaggle/input/imgps3k-yfcc4k-cleaned labeled_images=2997


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

CHECKPOINT rows=2905 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 2905/2997 elapsed=844.3s


CHECKPOINT rows=2930 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 2930/2997 elapsed=852.1s


CHECKPOINT rows=2955 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 2955/2997 elapsed=861.1s


CHECKPOINT rows=2980 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 2980/2997 elapsed=870.6s


CHECKPOINT rows=2997 path=/content/img2gps3k_wedetect_eval/per_image.csv
PROPOSALS 2997/2997 elapsed=876.1s
CHECKPOINT rows=2997 path=/content/img2gps3k_wedetect_eval/per_image.csv
FINAL_SUMMARY
{
  "dataset": "lbgan2000/imgps3k-yfcc4k-cleaned",
  "evaluated_images": 2997,
  "failed_images": 0,
  "device": "cuda",
  "gpu": "Tesla T4",
  "wedetect_provider": "CUDAExecutionProvider",
  "config": {
    "score_threshold": 0.4,
    "iou_threshold": 0.7,
    "proposal_top_k": 1000,
    "layer": 23,
    "a": 1.0,
    "b": 0.0,
    "baseline_batch_size": 64,
    "proposal_batch_size": 8
  },
  "proposal_statistics": {
    "images_with_proposals": 2050,
    "images_without_proposals": 947,
    "total_unique_patch_masks": 5663,
    "mean_per_image": 1.8895562228895562,
    "median_per_image": 1.0,
    "max_per_image": 24
  },
  "metrics": {
    "baseline": {
      "mean_distance_km": 1763.001499960349,
      "median_distance_km": 241.78190985130797,
      "accuracy": {
        "acc_1_km": 0.1304

*File Operation*: `download` on `/content/img2gps3k_wedetect_eval/summary.json`

*File Operation*: `download` on `/content/img2gps3k_wedetect_eval/per_image.csv`